### Backtracking

[Video](https://youtu.be/j4tds6gD3fQ) - [Video Beispiele](https://youtu.be/4bzAqfnZDB8) -
[Aufgabenvideo](https://youtu.be/H_9N8_OGHso)

Backtracking wenden wir an, wenn alle möglichen Lösungen für ein Problem gefragt sind. Wir erweitern schrittweise
Teillösungen zu einer Gesamtlösung. Wenn wir in eine Sackgasse geraten sind oder eine Lösung gefunden haben, nehmen wir
unsere letzte Entscheidung zurück und probieren weitere Varianten aus.

Beispiel: Wir wollen alle Permutationen von [1, 2, 3] finden, also alle Anordnungen der drei Zahlen.

<img src='permutation.png' width='800'>

Der Baum zeigt alle Entscheidungen. Wir gehen ihn von links nach rechts durch: Zuerst wählen wir 1, dann 2, dann 3 und
haben die Lösung [1, 2, 3]. Dann nehmen wir die 3 zurück, die 2 auch, probieren an ihrer Stelle die 3 und kommen zu
[1, 3, 2]. Und so weiter.

Backtracking ist anwendbar, wenn wir eine Lösung schrittweise aufbauen können und wenn wir für eine Teillösung die
möglichen Kandidaten aufzählen können, die die Teillösung erweitern.

- Eine **Teillösung** `a` ist eine Liste mit den bisher getroffenen Entscheidungen.
- Die **Kandidaten** sind die Möglichkeiten für die nächste Entscheidung.
- Ein Kandidat ist **gut**, wenn er die Teillösung sinnvoll erweitert, also nicht schon jetzt in eine Sackgasse führt.

Beim Backtracking wird eine einzige Teillösung immer wieder verändert. Erst wenn eine Lösung gefunden ist, wird eine
Kopie gesichert.

#### Muster für das Backtracking

In [1]:
def back(a):
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            # ggf. weitere Operationen
            back(a)
            # ggf. weitere Operationen rückgängig machen
            a.pop()

Im konkreten Fall schreiben wir drei Funktionen:

- `isSolution(a)`: `True`, wenn die Teillösung `a` eine Lösung ist,
- `candidates(a)`: die Liste der Entscheidungen, die die Teillösung `a` um eine Stufe erweitern können,
- `isGood(cand, a)`: `True`, wenn die Entscheidung `cand` die Teillösung `a` sinnvoll erweitert.

Aufruf: `solutions = []`, dann `back([])`, danach stehen alle Lösungen in `solutions`.

`back` ruft sich selbst auf. Wichtig ist, was **nach** dem Aufruf passiert: Mit `a.pop()` wird die Entscheidung wieder
zurückgenommen, damit der nächste Kandidat an dieselbe Stelle kommen kann. Das ist das „Zurückgehen“, das dem
Verfahren den Namen gibt.

#### Permutationen

Gesucht sind alle Permutationen von 1 bis n. Wir können eine Lösung in n Schritten aufbauen. In jedem Schritt stehen
uns die Kandidaten 1 bis n zur Verfügung. Ein Kandidat ist gut, wenn er noch nicht in der Teillösung vorkommt.

In [2]:
def isSolution(a):
    return len(a) == n

def candidates(a):
    return list(range(1, n + 1))

def isGood(cand, a):
    return cand not in a

n = 3
solutions = []
back([])
print(solutions)

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


Damit wir sehen, in welcher Reihenfolge `back` aufgerufen wird, geben wir das Argument jedes Aufrufs aus. Das
entspricht dem Durchlaufen des Baums im Bild oben.

In [3]:
def back_ausgabe(a):
    print('back', a)
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            back_ausgabe(a)
            a.pop()

solutions = []
back_ausgabe([])

back []
back [1]
back [1, 2]
back [1, 2, 3]
back [1, 3]
back [1, 3, 2]
back [2]
back [2, 1]
back [2, 1, 3]
back [2, 3]
back [2, 3, 1]
back [3]
back [3, 1]
back [3, 1, 2]
back [3, 2]
back [3, 2, 1]


`back` wird 16-mal aufgerufen: einmal für die leere Liste, dreimal mit einer, sechsmal mit zwei und sechsmal mit drei
Zahlen.

##### Warum `a.copy()`?

Würden wir `a` selbst in `solutions` speichern, stünde dort sechsmal **dieselbe** Liste. Sie wird danach weiter
verändert und ist am Ende leer:

In [4]:
def back_ohne_copy(a):
    if isSolution(a):
        solutions.append(a)                  # Fehler: keine Kopie
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            back_ohne_copy(a)
            a.pop()

solutions = []
back_ohne_copy([])
print(solutions)

[[], [], [], [], [], []]


Wir fassen den Code des Musters für die Permutationen zusammen. Das geht in einfachen Fällen gut:

In [5]:
def back(a):
    if len(a) == n:
        solutions.append(a.copy())
        return
    for cand in range(1, n + 1):
        if cand not in a:
            a.append(cand)
            back(a)
            a.pop()

n = 3
solutions = []
back([])
print(solutions)

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


#### 8-Damen-Problem

Wie viele Möglichkeiten gibt es, 8 Damen auf einem Schachbrett so zu positionieren, dass sie sich gegenseitig nicht
bedrohen? Eine Dame bedroht alle Felder in ihrer Zeile, ihrer Spalte und ihren beiden Diagonalen.

- Die Frage wurde 1848 in der Berliner Schachzeitung veröffentlicht.
- 1850 wurde erstmals die richtige Antwort gegeben: 92.

Wir überlegen uns das Vorgehen an dem kleineren 4-Damen-Problem. Wir können eine mögliche Lösung schrittweise
aufbauen, indem wir nacheinander entscheiden, in welche Spalte wir die Damen setzen. Wir beginnen mit der Dame ganz
oben (Zeile 0) und gehen dann schrittweise nach unten. In jede Zeile kommt genau eine Dame.

<img src='damen4.png'>

Wir müssen insgesamt 4 Entscheidungen treffen. Die Teillösung `a = [1, 3]` bedeutet: Die Dame in Zeile 0 steht in
Spalte 1, die Dame in Zeile 1 in Spalte 3. Für die nächste Zeile prüfen wir für jeden Kandidaten 0, 1, 2, 3, ob er die
Teillösung sinnvoll erweitert.

<img src='damen5.png' width='600'>

Die neue Dame kommt in Zeile `len(a)` und Spalte `cand`. Sie bedroht die Dame in Zeile `i` und Spalte `a[i]`,

- wenn beide in derselben Spalte stehen: `a[i] == cand`,
- wenn beide auf einer Diagonalen stehen: Dann ist der Abstand der Zeilen gleich dem Abstand der Spalten,
  `len(a) - i == abs(cand - a[i])`.

<img src='damen6.png'>

Am 8-Damen-Problem: Mit `a = [0, 6, 3, 5, 7]` soll die nächste Dame in Spalte 2. Für die Dame in Zeile 1 ist der
Zeilenabstand $5 - 1 = 4$ und der Spaltenabstand $|2 - 6| = 4$. Die beiden stehen auf einer Diagonalen.

<img src='damen2.png' width='500'>

In [6]:
def isSolution(a):
    return len(a) == anzahl_damen

def candidates(a):
    return list(range(anzahl_damen))

def isGood(cand, a):
    for i in range(len(a)):
        if a[i] == cand:
            return False
        if len(a) - i == abs(cand - a[i]):
            return False
    return True

def printLoesung(a):
    for i in range(anzahl_damen):
        for j in range(anzahl_damen):
            print('X' if a[i] == j else '.', end=' ')
        print()
    print()

def back(a):
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            back(a)
            a.pop()

anzahl_damen = 4
solutions = []
back([])
print(solutions)
print('Anzahl Lösungen:', len(solutions))
for k, a in enumerate(solutions):
    print(f'Lösung {k + 1}:')
    printLoesung(a)

[[1, 3, 0, 2], [2, 0, 3, 1]]
Anzahl Lösungen: 2
Lösung 1:
. X . . 
. . . X 
X . . . 
. . X . 

Lösung 2:
. . X . 
X . . . 
. . . X 
. X . . 



Und jetzt das klassische 8-Damen-Problem:

In [7]:
anzahl_damen = 8
solutions = []
back([])
print('Anzahl Lösungen:', len(solutions))
printLoesung(solutions[0])

Anzahl Lösungen: 92
X . . . . . . . 
. . . . X . . . 
. . . . . . . X 
. . . . . X . . 
. . X . . . . . 
. . . . . . X . 
. X . . . . . . 
. . . X . . . . 



Wie viel Arbeit spart das Backtracking? Ohne Prüfung müsste man jede Dame in jede der 8 Spalten setzen, das sind
$8^8 = 16777216$ Stellungen. Wir zählen die Aufrufe von `back`:

In [8]:
def back_zaehlen(a):
    global aufrufe
    aufrufe += 1
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            back_zaehlen(a)
            a.pop()

for anzahl_damen in range(1, 11):
    solutions = []
    aufrufe = 0
    back_zaehlen([])
    print(f'{anzahl_damen:2} Damen: {len(solutions):4} Lösungen, {aufrufe:6} Aufrufe von back')

 1 Damen:    1 Lösungen,      2 Aufrufe von back
 2 Damen:    0 Lösungen,      3 Aufrufe von back
 3 Damen:    0 Lösungen,      6 Aufrufe von back
 4 Damen:    2 Lösungen,     17 Aufrufe von back
 5 Damen:   10 Lösungen,     54 Aufrufe von back
 6 Damen:    4 Lösungen,    153 Aufrufe von back
 7 Damen:   40 Lösungen,    552 Aufrufe von back
 8 Damen:   92 Lösungen,   2057 Aufrufe von back
 9 Damen:  352 Lösungen,   8394 Aufrufe von back


10 Damen:  724 Lösungen,  35539 Aufrufe von back


Für 8 Damen sind es nur 2057 Aufrufe, weil `isGood` jede Sackgasse sofort erkennt und den ganzen Teilbaum darunter
gar nicht erst aufbaut. Für 2 und 3 Damen gibt es keine Lösung.

### Weitere Beispiele

Alle folgenden Probleme lösen wir mit demselben Muster. Wir müssen jeweils überlegen: Wie sieht eine Teillösung aus,
welches sind die Kandidaten, und wann ist ein Kandidat gut?

#### Lagerfeuer in der Jugendherberge

In der Jugendherberge organisiert man jeden Abend ein Lagerfeuer. Am ersten Abend setzen sich alle Jugendlichen rund
ums Feuer. Am zweiten Abend nehmen sie so Platz, dass die, die am ersten Abend Nachbarn waren, diesmal nicht
nebeneinandersitzen. Bestimme alle Möglichkeiten, wie sie sich am zweiten Abend setzen können. Wir nehmen an, dass sie
am ersten Abend der Reihe nach mit 1, 2, 3, ..., n nummeriert sind.

<img src='lagerfeuer.png' width='800'>

Die Teillösung ist die Liste der Personen, die schon sitzen, im Uhrzeigersinn. Am ersten Abend waren zwei Personen
Nachbarn, wenn sich ihre Nummern um 1 unterscheiden, und auch 1 und n, denn sie sitzen im Kreis. Ein Kandidat ist gut,
wenn er noch nicht sitzt und sein Abstand zum letzten Element der Teillösung weder 1 noch n - 1 ist.

<img src='lagerfeuer1.png' width='300'>

Weil die Personen im Kreis sitzen, sind am Ende auch der Erste und der Letzte der Liste Nachbarn. Das prüfen wir in
`isSolution`.

In [9]:
def isSolution(a):
    return len(a) == n and abs(a[-1] - a[0]) not in (1, n - 1)

def candidates(a):
    return list(range(1, n + 1))

def isGood(cand, a):
    if cand in a:
        return False
    if len(a) > 0 and abs(a[-1] - cand) in (1, n - 1):
        return False
    return True

def back(a):
    if len(a) == n:
        if isSolution(a):
            solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            back(a)
            a.pop()

for n in [5, 6]:
    solutions = []
    back([])
    print(f'n = {n}: {len(solutions)} Möglichkeiten, z.B.', *solutions[:3])

n = 5: 10 Möglichkeiten, z.B. [1, 3, 5, 2, 4] [1, 4, 2, 5, 3] [2, 4, 1, 3, 5]
n = 6: 36 Möglichkeiten, z.B. [1, 3, 5, 2, 6, 4] [1, 3, 6, 4, 2, 5] [1, 4, 2, 6, 3, 5]


Hier sind alle `n` Personen platziert, wenn `len(a) == n` ist, aber nur die Sitzordnungen, bei denen auch Erster und
Letzter keine alten Nachbarn sind, werden gespeichert. Ohne diese Prüfung würde das Programm für n = 6 fälschlich
60 statt 36 Möglichkeiten finden. Bei n = 5 fällt der Fehler zufällig nicht auf.

#### Referate verteilen

Die Erdkunde-Lehrerin möchte, dass die Schülerinnen und Schüler Referate halten, und verteilt die Themen. Sie hat für
die n Schülerinnen und Schüler m Themen vorbereitet ($m \le n$), von denen jedes mindestens einmal bearbeitet werden
soll. Welche Möglichkeiten gibt es, die Themen zu verteilen?

<img src='referate.png'>

Die Teillösung ist die Liste der Themen der ersten Schülerinnen und Schüler. Kandidaten sind die Themen 1 bis m. Hier
schauen wir voraus: Ein Kandidat ist nur gut, wenn danach noch genug Schülerinnen und Schüler übrig sind, um alle noch
fehlenden Themen zu bekommen.

In [10]:
def isSolution(a):
    return len(a) == n

def candidates(a):
    return list(range(1, m + 1))

def isGood(cand, a):
    vergeben = len(set(a) | {cand})          # verschiedene Themen inklusive cand
    fehlen = m - vergeben                     # Themen, die noch niemand hat
    uebrig = n - len(a) - 1                   # Schüler, die noch kein Thema haben
    return fehlen <= uebrig

def back(a):
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            back(a)
            a.pop()

n = 3
m = 2
solutions = []
back([])
print(f'Es gibt {len(solutions)} Möglichkeiten:')
for x in solutions:
    print(*x)

Es gibt 6 Möglichkeiten:
1 1 2
1 2 1
1 2 2
2 1 1
2 1 2
2 2 1


`set(a)` ist die Menge der Themen in `a`, jedes Thema kommt darin nur einmal vor. `|` vereinigt zwei Mengen.

#### Das Haus vom Nikolaus

Wie viele Möglichkeiten gibt es, das Haus vom Nikolaus beginnend von der linken unteren Ecke zu zeichnen, ohne den
Stift abzusetzen und ohne eine Linie doppelt zu ziehen?

Das Haus hat 5 Knoten und 8 Kanten. Wir beginnen immer mit Knoten 0. Dann treffen wir 8 Entscheidungen: Welches ist
unser nächster Knoten? Ein Kandidat ist gut, wenn es eine Kante zu ihm gibt und sie noch nicht benutzt wurde. Dazu
merken wir uns im Dictionary `used`, welche Kanten verbraucht sind.

<img src='nikolaus.png' width='600'>

In [11]:
def isSolution(a):
    return len(a) == 9

def candidates(a):
    return list(range(5))

def isGood(cand, a):
    if (a[-1], cand) not in edges:
        return False
    return not used[(a[-1], cand)]

def back(a):
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            used[(a[-1], cand)] = True       # diese Kante ist verbraucht
            used[(cand, a[-1])] = True
            a.append(cand)
            back(a)
            a.pop()
            used[(a[-1], cand)] = False      # diese Kante ist wieder frei
            used[(cand, a[-1])] = False

# jede Kante in beiden Richtungen
edges = [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 4), (4, 3), (4, 0), (0, 4),
         (1, 3), (3, 1), (0, 3), (3, 0), (1, 4), (4, 1)]
used = {e: False for e in edges}

solutions = []
back([0])
print(len(solutions), 'Möglichkeiten')
for sol in solutions[:5]:
    print(*sol)

44 Möglichkeiten
0 1 2 3 0 4 1 3 4
0 1 2 3 0 4 3 1 4
0 1 2 3 1 4 0 3 4
0 1 2 3 1 4 3 0 4
0 1 2 3 4 0 3 1 4


Hier sehen wir die Stelle „ggf. weitere Operationen“ aus dem Muster: Vor dem rekursiven Aufruf wird die Kante als
benutzt markiert, danach wieder freigegeben.

#### Sudoku

Das Vorgehen überlegen wir uns an einem 4 × 4-Sudoku. In jeder Zeile, jeder Spalte und jedem der vier 2 × 2-Quadrate
muss jede Zahl von 1 bis 4 genau einmal vorkommen. Ein ordentliches Sudoku hat nur eine Lösung. Für den Fall, dass es
doch mehrere Lösungen gibt, soll unser Programm alle finden.

<img src='sudoku1.png' width='500'>

In [12]:
import numpy as np

data = '''
.2..
1.4.
...3
3...
'''

zeilen = data.strip().split()
n = len(zeilen)                  # Größe des Sudokus
sn = int(np.sqrt(n))             # Größe der Quadrate
grid = np.zeros((n, n), dtype='int')
for x in range(n):
    for y in range(n):
        grid[x, y] = int(zeilen[x][y]) if zeilen[x][y].isdigit() else 0
print(grid)

[[0 2 0 0]
 [1 0 4 0]
 [0 0 0 3]
 [3 0 0 0]]


In der Liste `leer` speichern wir die Koordinaten (Zeile, Spalte) der Leerstellen. Eine Teillösung gibt der Reihe nach
für die ersten Leerstellen eine Zahl an. Bei diesem Sudoku sind 11 Entscheidungen zu treffen.

In [13]:
leer = []
for x in range(n):
    for y in range(n):
        if grid[x, y] == 0:
            leer.append((x, y))
print(leer)

[(0, 0), (0, 2), (0, 3), (1, 1), (1, 3), (2, 0), (2, 1), (2, 2), (3, 1), (3, 2), (3, 3)]


Ein Kandidat für die nächste Leerstelle `(x, y)` ist gut, wenn er weder in Zeile `x` noch in Spalte `y` noch in dem
Quadrat vorkommt, in dem die Leerstelle liegt. Die linke obere Ecke des Quadrats ist `(x - x % sn, y - y % sn)`.

<img src='sudoku2.png' width='800'>

Damit `isGood` die schon gesetzten Zahlen sieht, tragen wir jeden Kandidaten auch in `grid` ein und nehmen ihn nach dem
rekursiven Aufruf wieder heraus.

In [14]:
def isSolution(a):
    return len(a) == len(leer)

def candidates(a):
    return list(range(1, n + 1))

def isGood(cand, a):
    x, y = leer[len(a)]              # die nächste zu füllende Leerstelle
    x1 = x - x % sn                  # die linke obere Ecke
    y1 = y - y % sn                  # des zugehörigen Quadrats
    quadrat = grid[x1:x1 + sn, y1:y1 + sn].flatten()
    return cand not in grid[x, :] and cand not in grid[:, y] and cand not in quadrat

def back(a):
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            grid[leer[len(a) - 1]] = cand
            back(a)
            grid[leer[len(a) - 1]] = 0
            a.pop()

def zeige_loesungen():
    print(f'Anzahl der Lösungen: {len(solutions)}')
    for solution in solutions:
        fertig = grid.copy()
        for k in range(len(leer)):
            fertig[leer[k]] = solution[k]
        print(fertig)

solutions = []
back([])
zeige_loesungen()

Anzahl der Lösungen: 1
[[4 2 3 1]
 [1 3 4 2]
 [2 4 1 3]
 [3 1 2 4]]


Damit wir beliebige Sudokus lösen können, fassen wir die Vorbereitung in einer Funktion zusammen. Ein 9 × 9-Sudoku
dauert etwas länger:

In [15]:
def loese_sudoku(data):
    global grid, leer, n, sn, solutions
    zeilen = data.strip().split()
    n = len(zeilen)
    sn = int(np.sqrt(n))
    grid = np.zeros((n, n), dtype='int')
    for x in range(n):
        for y in range(n):
            grid[x, y] = int(zeilen[x][y]) if zeilen[x][y].isdigit() else 0
    leer = []
    for x in range(n):
        for y in range(n):
            if grid[x, y] == 0:
                leer.append((x, y))
    solutions = []
    back([])
    zeige_loesungen()

loese_sudoku('''
...5.8936
.....3.54
9354...78
679.8...5
.......8.
.1..74..2
7..25...3
4.....72.
...1.....
''')

Anzahl der Lösungen: 1
[[1 4 7 5 2 8 9 3 6]
 [8 6 2 7 9 3 1 5 4]
 [9 3 5 4 6 1 2 7 8]
 [6 7 9 3 8 2 4 1 5]
 [3 2 4 9 1 5 6 8 7]
 [5 1 8 6 7 4 3 9 2]
 [7 9 1 2 5 6 8 4 3]
 [4 5 6 8 3 9 7 2 1]
 [2 8 3 1 4 7 5 6 9]]


#### Alle Pfade in einem Graphen

In einem gerichteten Graphen ohne Kreise (DAG, directed acyclic graph) wollen wir alle Pfade von einem Startknoten zu
einem Zielknoten bestimmen. Die Teillösung ist der bisherige Pfad, die Kandidaten sind die Nachfolger des letzten
Knotens. Weil es keine Kreise gibt, ist jeder Kandidat gut.

<img src='graph2.png' width='301'>

Hier fassen wir den Code gleich zusammen. `back` steht **innerhalb** von `allPaths` und kann deshalb `G`, `end` und
`solutions` benutzen.

In [16]:
def allPaths(G, start, end):
    '''returns: Liste aller Pfade von start nach end im DAG G'''
    def back(a):
        if a[-1] == end:
            solutions.append(a.copy())
            return
        for cand in sorted(G[a[-1]]):
            a.append(cand)
            back(a)
            a.pop()

    solutions = []
    back([start])
    return solutions

G = {0: [4, 3, 1], 1: [3, 2, 4], 2: [3], 3: [4], 4: []}
allPaths(G, 0, 4)

[[0, 1, 2, 3, 4], [0, 1, 3, 4], [0, 1, 4], [0, 3, 4], [0, 4]]

<img src='graph1.png' width='400'>

In [17]:
G1 = {1: {2, 4, 7}, 2: {3}, 3: set(), 4: {5}, 5: {6}, 6: set(), 7: {4, 8}, 8: {5, 9}, 9: {5}}
allPaths(G1, 1, 6)

[[1, 4, 5, 6], [1, 7, 4, 5, 6], [1, 7, 8, 5, 6], [1, 7, 8, 9, 5, 6]]

#### Alle Kreise in einem Graphen

Ein **einfacher Kreis** ist ein Pfad, der zu seinem Startknoten zurückführt und dabei keinen Knoten doppelt besucht.
Mit Backtracking finden wir alle einfachen Kreise, die von einem Knoten `v` ausgehen. Die Teillösung beginnt mit `v`.
Ein Kandidat ist gut, wenn er noch nicht im Pfad vorkommt, nur `v` selbst darf am Ende noch einmal kommen. Die Nachfolger
gehen wir alphabetisch durch.

<img src='kreise1.png' width='300'>

In [18]:
def kreise(G, v):
    '''returns: Liste aller einfachen Kreise von v nach v'''
    def back(a):
        if len(a) > 1 and a[-1] == a[0]:
            solutions.append(a.copy())
            return
        for cand in sorted(G[a[-1]]):
            if cand not in a[1:]:
                a.append(cand)
                back(a)
                a.pop()

    solutions = []
    back([v])
    return solutions

G = {'a': {'b'}, 'b': {'c'}, 'c': {'d', 'f'}, 'd': {'e'}, 'e': {'f'}, 'f': {'a', 'c', 'e'}}
kreise(G, 'f')

[['f', 'a', 'b', 'c', 'd', 'e', 'f'],
 ['f', 'a', 'b', 'c', 'f'],
 ['f', 'c', 'd', 'e', 'f'],
 ['f', 'c', 'f'],
 ['f', 'e', 'f']]

Starten wir bei jedem Knoten, finden wir jeden Kreis mehrmals, z.B. f, c, f und c, f, c. Damit Kreise nicht doppelt
gezählt werden, beschreiben wir jeden Kreis durch seine sortierte Liste von Kanten. Eine Kante schreiben wir als String
aus zwei Buchstaben, z.B. `'cf'` für die Kante von c nach f.

In [19]:
def getKanten(kreis):
    tmp = []
    for i in range(len(kreis) - 1):
        tmp.append(kreis[i] + kreis[i + 1])
    tmp.sort()
    return tmp

def alleKreise(G):
    tmp = []
    for v in sorted(G):
        for kreis in kreise(G, v):
            kanten = getKanten(kreis)
            if kanten not in tmp:
                tmp.append(kanten)
    return tmp

for kanten in alleKreise(G):
    print(*kanten)

ab bc cd de ef fa
ab bc cf fa
cd de ef fc
cf fc
ef fe


#### Wortumformungen

Wir dürfen das Wort DACH verändern:

- Eine Umstellung vom Typ ABCD → BCDA (der erste Buchstabe wandert nach hinten) kostet 1.
- Eine Umstellung vom Typ ABCD → BADC (vertauscht werden die Buchstaben 1 und 2 sowie 3 und 4) kostet 2.

Wie viele Folgen von Umstellungen gibt es, die von DACH wieder zu DACH führen und genau 10 kosten?

Eine Umstellung beschreiben wir durch die Liste der Positionen, aus denen die neuen Buchstaben kommen: BCDA ist
`[1, 2, 3, 0]`, BADC ist `[1, 0, 3, 2]`. Die Teillösung ist eine Liste von Paaren (Wort, bisherige Kosten). Ein Kandidat
ist gut, wenn die Kosten 10 nicht überschreiten.

In [20]:
def apply(p, s):
    '''wendet die Umstellung p auf das Wort s an'''
    ergebnis = ''
    for i in p:
        ergebnis += s[i]
    return ergebnis

perms = [[1, 2, 3, 0], [1, 0, 3, 2]]
kosten = [1, 2]
print(apply(perms[0], 'DACH'), apply(perms[1], 'DACH'))

ACHD ADHC


In [21]:
def isSolution(a):
    return a[-1] == (start, zielkosten)

def candidates(a):
    wort, k = a[-1]
    return [(apply(perms[i], wort), k + kosten[i]) for i in range(len(perms))]

def isGood(cand, a):
    return cand[1] <= zielkosten

def back(a):
    if isSolution(a):
        solutions.append(a.copy())
        return
    for cand in candidates(a):
        if isGood(cand, a):
            a.append(cand)
            back(a)
            a.pop()

start = 'DACH'
zielkosten = 10
solutions = []
back([(start, 0)])
print(f'Es gibt {len(solutions)} Möglichkeiten, z.B.:')
print(*solutions[0])

Es gibt 18 Möglichkeiten, z.B.:
('DACH', 0) ('ACHD', 1) ('CHDA', 2) ('HDAC', 3) ('DACH', 4) ('ACHD', 5) ('CADH', 7) ('ADHC', 8) ('DACH', 10)


Hier wird `isSolution` auch dann geprüft, wenn DACH schon nach weniger als 10 erreicht ist. Dann ist es keine Lösung,
und es geht weiter, bis die Kosten 10 überschritten würden.

### So geht man bei einem neuen Problem vor

1. **Teillösung:** Wie sieht eine Teillösung aus? Meist eine Liste der bisherigen Entscheidungen.
2. **Lösung:** Wann ist eine Teillösung eine vollständige Lösung? Das wird `isSolution`.
3. **Kandidaten:** Welche Möglichkeiten gibt es für die nächste Entscheidung? Das wird `candidates`.
4. **Gut:** Welche Kandidaten führen nicht sofort in eine Sackgasse? Das wird `isGood`. Je früher eine Sackgasse
   erkannt wird, desto weniger muss durchsucht werden.
5. **Zusätzliche Daten:** Muss neben `a` noch etwas mitgeführt werden (wie `used` beim Nikolaus oder `grid` beim
   Sudoku)? Dann wird es vor dem rekursiven Aufruf geändert und danach zurückgesetzt.

#### Zum Weiterüben

Codingame:

- [Sudoku-Solver](https://www.codingame.com/training/medium/sudoku-solver)
- [Finish The Eight Queens](https://www.codingame.com/training/medium/finish-the-eight-queens)